# S5 main Bangla — smoke test and resumable chunks (Kaggle T4)

**RUNNER ONLY.** All experimental logic lives in `src/eval/`; this notebook only clones, installs, restores checkpoints, invokes checked subprocesses, and exports files.

This runner resumes the verified seed-42 checkpoint through base case 139 and runs the final seed-42 base cases 140–179. Attach `s5-bn-checkpoint-seed42-through-base139`; its 1,400 completed condition-cases are preserved and the next 400 condition-cases complete seed 42. Requirements: Internet ON, the Kaggle T4×2 allocation (the frozen Writer uses `cuda:0` only), the `google/gemma-3-12b-it` Kaggle Model input, one dataset containing `bn_clean.csv`, the checkpoint dataset, and a Kaggle secret named `GOOGLE_API_KEY`. **Do not add Verifier-B**: generation must not be able to load it.


In [ ]:
from pathlib import Path
import os, subprocess
RUNNER_COMMIT = '22124a816e5ecc9d6fa59c957bd939cfd311a28a'
REPO = Path('/kaggle/working/s5_repo_22124a8')
if not (REPO / '.git').is_dir():
    subprocess.run(['git','clone','-q','https://github.com/alphapie77/BSc_Thesis.git',str(REPO)], check=True)
subprocess.run(['git','-C',str(REPO),'checkout','--detach',RUNNER_COMMIT], check=True)
actual = subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'], text=True).strip()
assert actual == RUNNER_COMMIT, (actual, RUNNER_COMMIT)
os.chdir(REPO)
subprocess.run(['git','log','--oneline','-1'], check=True)


In [ ]:
# Experiments run in fresh subprocesses, so no kernel restart is needed.
import subprocess, sys
subprocess.run([sys.executable,'-m','pip','uninstall','-y','sklearn-compat'], check=False)
subprocess.run([sys.executable,'-m','pip','install','-q','-U','scikit-learn==1.9.0','transformers==5.15.0','sentence-transformers==5.6.1','accelerate','bitsandbytes','chromadb','pyyaml','joblib','pytest'], check=True)
gate = "import sklearn,transformers; print('sklearn',sklearn.__version__,'transformers',transformers.__version__); assert sklearn.__version__=='1.9.0'; assert transformers.__version__=='5.15.0'"
subprocess.run([sys.executable,'-c',gate], check=True)


In [ ]:
from pathlib import Path
import glob, os, shutil, subprocess, sys
from kaggle_secrets import UserSecretsClient
REPO = Path('/kaggle/working/s5_repo_22124a8')
os.chdir(REPO)
clean = glob.glob('/kaggle/input/**/bn_clean.csv', recursive=True)
assert len(clean) == 1, f'expected one bn_clean.csv input, found: {clean}'
Path('data/cleaned').mkdir(parents=True, exist_ok=True)
shutil.copy2(clean[0], 'data/cleaned/bn_clean.csv')
gemma = [str(Path(p).parent) for p in glob.glob('/kaggle/input/**/config.json', recursive=True) if 'gemma-3-12b-it' in p.lower()]
assert len(gemma) == 1, f'expected one Gemma input, found: {gemma}'
MODEL_PATH = gemma[0]
os.environ['GOOGLE_API_KEY'] = UserSecretsClient().get_secret('GOOGLE_API_KEY')
assert os.environ['GOOGLE_API_KEY'], 'Kaggle secret GOOGLE_API_KEY is empty'
print('Gemma:', MODEL_PATH)
print('Google key: present (never printed)')


In [ ]:
# Restore a previous exported checkpoint dataset if attached. Exact basenames only.
from pathlib import Path
import glob, os, shutil, subprocess, sys
REPO = Path('/kaggle/working/s5_repo_22124a8')
os.chdir(REPO)
restore = {
    's5_main_bn_calls.jsonl': Path('data/generated/s5_main_bn_calls.jsonl'),
    's5_main_bn_gemini_calls.jsonl': Path('data/generated/s5_main_bn_gemini_calls.jsonl'),
    's5_main_bn_gemini_transport_failures.jsonl': Path('data/generated/s5_main_bn_gemini_transport_failures.jsonl'),
    's5_main_bn_gemma4_feedback_v1_superseded_calls.jsonl': Path('data/generated/s5_main_bn_gemma4_feedback_v1_superseded_calls.jsonl'),
    's5_main_bn_gemma4_feedback_v1_superseded_judge_calls.jsonl': Path('data/generated/s5_main_bn_gemma4_feedback_v1_superseded_judge_calls.jsonl'),
    's5_main_bn_gemma4_feedback_v1_superseded_transport_failures.jsonl': Path('data/generated/s5_main_bn_gemma4_feedback_v1_superseded_transport_failures.jsonl'),
    's5_main_bn_gemma4_feedback_v1_superseded_cases.jsonl': Path('results/s5_main_bn_gemma4_feedback_v1_superseded_cases.jsonl'),
    's5_main_bn_cases.jsonl': Path('results/s5_main_bn_cases.jsonl'),
    's5_main_bn_gemini36_superseded_calls.jsonl': Path('data/generated/s5_main_bn_gemini36_superseded_calls.jsonl'),
    's5_main_bn_gemini36_superseded_writer_calls.jsonl': Path('data/generated/s5_main_bn_gemini36_superseded_writer_calls.jsonl'),
    's5_main_bn_gemini36_superseded_cases.jsonl': Path('results/s5_main_bn_gemini36_superseded_cases.jsonl'),
}
for name, dst in restore.items():
    found = glob.glob('/kaggle/input/**/' + name, recursive=True)
    if found:
        assert len(found) == 1, f'multiple checkpoint candidates for {name}: {found}'
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(found[0], dst)
        print('restored', name, dst.stat().st_size, 'bytes')
    else:
        print('new archive:', name)
subprocess.run([sys.executable,'src/eval/migrate_s5_gemma4_feedback_enum.py','--config','configs/s5_main_bn.yaml'], check=True)


In [ ]:
# Runtime/API/tokenizer gates and then CPU scientific gates, all before the 12B model loads.
from pathlib import Path
import glob, os, subprocess, sys
from kaggle_secrets import UserSecretsClient
REPO = Path('/kaggle/working/s5_repo_22124a8')
gemma = [str(Path(p).parent) for p in glob.glob('/kaggle/input/**/config.json', recursive=True) if 'gemma-3-12b-it' in p.lower()]
assert len(gemma) == 1, f'expected one Gemma input, found: {gemma}'
MODEL_PATH = gemma[0]
os.environ['GOOGLE_API_KEY'] = UserSecretsClient().get_secret('GOOGLE_API_KEY')
os.chdir(REPO)
subprocess.run([sys.executable,'src/eval/preflight_s5_kaggle.py','--config','configs/s5_main_bn.yaml','--model-path',MODEL_PATH], check=True)
subprocess.run([sys.executable,'src/agents/build_index.py','--config','configs/s4_index.yaml','--index-only'], check=True)
subprocess.run([sys.executable,'-m','pytest','tests/test_s5_contract.py','tests/test_s5_engine.py','tests/test_s5_gemini_judge.py','tests/test_s5_prompts.py','tests/test_s5_kaggle_preflight.py','tests/test_s5_checkpoint_migration.py','tests/test_s5_feedback_enum_migration.py','-q'], check=True)
subprocess.run([sys.executable,'src/eval/run_s5_main_bn.py','--config','configs/s5_main_bn.yaml','--dry-run'], check=True)


## Smoke test

The smoke is already verified and disabled for this resume run. Every completed call is appended immediately, so an interruption is resumable. Hosted Gemma-4-26B-A4B is the explicitly same-family judge condition; Verifier-B is absent.


In [ ]:
from pathlib import Path
import glob, os, subprocess, sys
from kaggle_secrets import UserSecretsClient
REPO = Path('/kaggle/working/s5_repo_22124a8')
gemma = [str(Path(p).parent) for p in glob.glob('/kaggle/input/**/config.json', recursive=True) if 'gemma-3-12b-it' in p.lower()]
assert len(gemma) == 1, f'expected one Gemma input, found: {gemma}'
MODEL_PATH = gemma[0]
os.environ['GOOGLE_API_KEY'] = UserSecretsClient().get_secret('GOOGLE_API_KEY')
os.chdir(REPO)
RUN_SMOKE = False
if RUN_SMOKE:
    subprocess.run([sys.executable,'src/eval/run_s5_main_bn.py','--config','configs/s5_main_bn.yaml','--model-path',MODEL_PATH,'--replicate-seed','42','--start','0','--limit','1'], check=True)
else:
    print('smoke disabled')


## Seed 42, final chunk — enabled

The verified checkpoint through base case 139 is attached. This run completes the final seed-42 bases 140–179 across all ten conditions. Reattaching the exported checkpoint then makes seed 43 resumable. Run only one Kaggle session at a time: the hosted-judge limiter enforces 90% of the observed 30 RPM / 16K TPM / 14.4K RPD account limits, but cannot see unrelated calls from another concurrent session.


In [ ]:
from pathlib import Path
import glob, os, subprocess, sys
from kaggle_secrets import UserSecretsClient
REPO = Path('/kaggle/working/s5_repo_22124a8')
gemma = [str(Path(p).parent) for p in glob.glob('/kaggle/input/**/config.json', recursive=True) if 'gemma-3-12b-it' in p.lower()]
assert len(gemma) == 1, f'expected one Gemma input, found: {gemma}'
MODEL_PATH = gemma[0]
os.environ['GOOGLE_API_KEY'] = UserSecretsClient().get_secret('GOOGLE_API_KEY')
os.chdir(REPO)
RUN_CHUNK = True
REPLICATE_SEED = 42
START_CASE = 140
N_CASES = 40
if RUN_CHUNK:
    subprocess.run([sys.executable,'src/eval/run_s5_main_bn.py','--config','configs/s5_main_bn.yaml','--model-path',MODEL_PATH,'--replicate-seed',str(REPLICATE_SEED),'--start',str(START_CASE),'--limit',str(N_CASES)], check=True)
else:
    print('full chunk disabled; smoke only')


In [ ]:
# Export an uploadable checkpoint after smoke or every chunk.
from pathlib import Path
import os, shutil, subprocess, sys
REPO = Path('/kaggle/working/s5_repo_22124a8')
os.chdir(REPO)
snapshot = 'results/env_snapshot_s5_bn_kaggle.json'
subprocess.run([sys.executable,'src/common/env_snapshot.py','--out',snapshot], check=True)
export_dir = Path('/kaggle/working/s5_checkpoint')
if export_dir.is_dir():
    shutil.rmtree(export_dir)  # prevent stale files entering a later export
export_dir.mkdir(parents=True, exist_ok=False)
files = [Path('data/generated/s5_main_bn_calls.jsonl'), Path('data/generated/s5_main_bn_gemini_calls.jsonl'), Path('data/generated/s5_main_bn_gemini_transport_failures.jsonl'), Path('data/generated/s5_main_bn_gemma4_feedback_v1_superseded_calls.jsonl'), Path('data/generated/s5_main_bn_gemma4_feedback_v1_superseded_judge_calls.jsonl'), Path('data/generated/s5_main_bn_gemma4_feedback_v1_superseded_transport_failures.jsonl'), Path('data/generated/s5_main_bn_gemini36_superseded_calls.jsonl'), Path('data/generated/s5_main_bn_gemini36_superseded_writer_calls.jsonl'), Path('results/s5_main_bn_cases.jsonl'), Path('results/s5_main_bn_gemma4_feedback_v1_superseded_cases.jsonl'), Path('results/s5_main_bn_gemini36_superseded_cases.jsonl'), Path('results/s5_main_bn_preflight.json'), Path(snapshot), Path('results/s4_index_manifest.json')]
for src in files:
    if src.is_file():
        shutil.copy2(src, export_dir / src.name)
        print('saved', src.name, src.stat().st_size, 'bytes')
archive = shutil.make_archive('/kaggle/working/s5_checkpoint', 'zip', export_dir)
assert archive.endswith('/s5_checkpoint.zip')
print('DOWNLOAD OR SAVE AS DATASET:', archive)
